# Organización de datos de jugadores por equipo

Este código recorre todas las subcarpetas dentro de `datos` buscando archivos `players.csv` descargados previamente. Para cada archivo encontrado:

- Se lee el CSV usando **pandas**.
- Se detecta automáticamente la columna que contiene el nombre del equipo del jugador.
- Si un jugador aparece en varios equipos (por ejemplo `"Equipo1, Equipo2"`), se **duplica su fila** para que aparezca en cada equipo por separado.
- Se crean subcarpetas `equipos` dentro de cada carpeta de liga/temporada.
- Se generan archivos CSV individuales por equipo, con nombres normalizados (sin espacios ni caracteres problemáticos), usando **pandas**.

De esta forma, se obtiene un dataset organizado **por equipo**, listo para análisis o procesamiento posterior, asegurando que cada jugador figure correctamente en todos los equipos en los que ha jugado.


In [1]:
import pandas as pd
import os

BASE_DIR = "datos"

# Recorre todas las subcarpetas y busca "players.csv"
for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if file == "players.csv":
            csv_path = os.path.join(root, file)
            print(f"\n📂 Procesando: {csv_path}")
            try:
                # Leer el archivo
                df = pd.read_csv(csv_path)

                # Detectar columna de equipo
                team_col = None
                for col in df.columns:
                    if "team" in col.lower():
                        team_col = col
                        break
                if team_col is None:
                    print(f"⚠️ No se encontró columna de equipo en {csv_path}")
                    continue

                # Separar jugadores con múltiples equipos
                df[team_col] = df[team_col].str.split(",")  # dividir por coma
                df = df.explode(team_col)  # crear una fila por cada equipo
                df[team_col] = df[team_col].str.strip()  # limpiar espacios

                # Crear carpeta de salida: subcarpeta "equipos"
                output_dir = os.path.join(root, "equipos")
                os.makedirs(output_dir, exist_ok=True)

                # Agrupar por equipo y guardar CSV por equipo
                for team, group in df.groupby(team_col):
                    filename = f"{team.replace('/', '_').replace(' ', '_')}.csv"
                    output_path = os.path.join(output_dir, filename)
                    group.to_csv(output_path, index=False)

                print(f"✅ Archivos guardados por equipo en: {output_dir}")

            except Exception as e:
                print(f"❌ Error procesando {csv_path}: {e}")



📂 Procesando: datos\Bundesliga\2020-2021\players.csv
✅ Archivos guardados por equipo en: datos\Bundesliga\2020-2021\equipos

📂 Procesando: datos\Bundesliga\2021-2022\players.csv
✅ Archivos guardados por equipo en: datos\Bundesliga\2021-2022\equipos

📂 Procesando: datos\Bundesliga\2022-2023\players.csv
✅ Archivos guardados por equipo en: datos\Bundesliga\2022-2023\equipos

📂 Procesando: datos\Bundesliga\2023-2024\players.csv
✅ Archivos guardados por equipo en: datos\Bundesliga\2023-2024\equipos

📂 Procesando: datos\Bundesliga\2024-2025\players.csv
✅ Archivos guardados por equipo en: datos\Bundesliga\2024-2025\equipos

📂 Procesando: datos\La-Liga\2020-2021\players.csv
✅ Archivos guardados por equipo en: datos\La-Liga\2020-2021\equipos

📂 Procesando: datos\La-Liga\2021-2022\players.csv
✅ Archivos guardados por equipo en: datos\La-Liga\2021-2022\equipos

📂 Procesando: datos\La-Liga\2022-2023\players.csv
✅ Archivos guardados por equipo en: datos\La-Liga\2022-2023\equipos

📂 Procesando: dat

# Carga de datos de jugadores por liga y temporada

Este código recorre todas las carpetas dentro de `datos`, donde previamente se han guardado archivos `players.csv` por liga y temporada, y carga los datos en **diccionarios de pandas** organizados de la siguiente manera:

- Se crean diccionarios individuales para cada liga: Bundesliga, La Liga, Serie A, Premier League y Ligue 1.
- Cada diccionario almacena los `DataFrame` de jugadores de cada temporada.
- Se construye un diccionario maestro `dfsAll` que agrupa todos los diccionarios de liga.
- Para cada archivo CSV encontrado:
  - Se lee usando **pandas**.
  - Se añaden las columnas `league` y `season` para mantener el contexto de los datos.
  - Se guarda el `DataFrame` en el diccionario correspondiente de la liga y temporada.
- Esto permite tener un **acceso organizado y rápido** a los datos de jugadores por liga y temporada, listo para análisis o procesamiento posterior.


In [2]:
import pandas as pd
import os

BASE_DIR = "datos"

# Diccionarios individuales por liga
dfsBundesliga = {}
dfsLa_Liga = {}
dfsSerie_A = {}
dfsPremier_League = {}
dfsLigue_1 = {}

# Diccionario maestro que los agrupa todos
dfsAll = {
    "Bundesliga": dfsBundesliga,
    "La-Liga": dfsLa_Liga,
    "Serie-A": dfsSerie_A,
    "Premier-League": dfsPremier_League,
    "Ligue-1": dfsLigue_1
}

# Recorrer todas las carpetas de ligas y temporadas
for league in os.listdir(BASE_DIR):
    league_path = os.path.join(BASE_DIR, league)
    if not os.path.isdir(league_path) or league not in dfsAll:
        continue

    for season in os.listdir(league_path):
        season_path = os.path.join(league_path, season)
        if not os.path.isdir(season_path):
            continue

        csv_path = os.path.join(season_path, "players.csv")
        if not os.path.exists(csv_path):
            continue

        try:
            df = pd.read_csv(csv_path)
            df["league"] = league
            df["season"] = season

            # Guardar dentro del diccionario correcto
            dfsAll[league][season] = df

            print(f"✅ Cargado {league} {season} ({len(df)} filas)")
        except Exception as e:
            print(f"⚠️ Error con {csv_path}: {e}")


✅ Cargado Bundesliga 2020-2021 (496 filas)
✅ Cargado Bundesliga 2021-2022 (511 filas)
✅ Cargado Bundesliga 2022-2023 (506 filas)
✅ Cargado Bundesliga 2023-2024 (494 filas)
✅ Cargado Bundesliga 2024-2025 (481 filas)
✅ Cargado La-Liga 2020-2021 (570 filas)
✅ Cargado La-Liga 2021-2022 (603 filas)
✅ Cargado La-Liga 2022-2023 (584 filas)
✅ Cargado La-Liga 2023-2024 (598 filas)
✅ Cargado La-Liga 2024-2025 (590 filas)
✅ Cargado Ligue-1 2020-2021 (575 filas)
✅ Cargado Ligue-1 2021-2022 (589 filas)
✅ Cargado Ligue-1 2022-2023 (585 filas)
✅ Cargado Ligue-1 2023-2024 (525 filas)
✅ Cargado Ligue-1 2024-2025 (542 filas)
✅ Cargado Premier-League 2020-2021 (524 filas)
✅ Cargado Premier-League 2021-2022 (537 filas)
✅ Cargado Premier-League 2022-2023 (554 filas)
✅ Cargado Premier-League 2023-2024 (570 filas)
✅ Cargado Premier-League 2024-2025 (562 filas)
✅ Cargado Serie-A 2020-2021 (594 filas)
✅ Cargado Serie-A 2021-2022 (608 filas)
✅ Cargado Serie-A 2022-2023 (577 filas)
✅ Cargado Serie-A 2023-2024 (5

Claves del diccionario dfsAll, que contiene todas las ligas

In [3]:
dfsAll.keys()

dict_keys(['Bundesliga', 'La-Liga', 'Serie-A', 'Premier-League', 'Ligue-1'])

Claves del diccionario de las ligas dentro del diccionario dfsAll

In [4]:
dfsAll["Bundesliga"].keys()

dict_keys(['2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025'])

Mostramos las primeras lineas del df LaLiga 2022-2023

In [5]:
df = dfsAll["La-Liga"]["2022-2023"]
df.head()

,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup,league,season
0,227,Robert Lewandowski,34,2862,23,25.939281,7,6.353777,135,36,2,1,F S,Barcelona,23,25.196005,36.746244,9.871429,La-Liga,2022-2023
1,2370,Karim Benzema,24,2056,19,23.952812,3,6.247547,107,48,1,0,F,Real Madrid,12,18.006583,25.885614,8.579490,La-Liga,2022-2023
2,866,Joselu,34,2989,16,15.871624,2,2.095152,93,22,2,0,F S,Espanyol,11,12.155238,13.920118,2.132708,La-Liga,2022-2023
3,2270,Antoine Griezmann,38,2853,15,11.751190,16,15.643586,111,85,2,0,F M S,Atletico Madrid,15,11.751190,30.893706,11.252617,La-Liga,2022-2023
4,2543,Borja Iglesias,35,2406,15,15.634133,3,2.921825,63,22,2,1,F S,Real Betis,10,11.174591,15.961691,3.046393,La-Liga,2022-2023


Mostramos el tamaño que tiene el df LaLiga 2023-2024

In [6]:
dfsAll["La-Liga"]["2023-2024"].shape

(598, 20)

Mostramos cuantos jugadores hay por liga y temporada

In [8]:
for league, seasons in dfsAll.items():
    for season, df in seasons.items():
        # Conteo de jugadores únicos por nombre
        unique_players = df['player_name'].nunique()
        print(f"{league} {season}: {unique_players} jugadores únicos")


Bundesliga 2020-2021: 496 jugadores únicos
Bundesliga 2021-2022: 511 jugadores únicos
Bundesliga 2022-2023: 506 jugadores únicos
Bundesliga 2023-2024: 494 jugadores únicos
Bundesliga 2024-2025: 481 jugadores únicos
La-Liga 2020-2021: 565 jugadores únicos
La-Liga 2021-2022: 601 jugadores únicos
La-Liga 2022-2023: 581 jugadores únicos
La-Liga 2023-2024: 597 jugadores únicos
La-Liga 2024-2025: 588 jugadores únicos
Serie-A 2020-2021: 593 jugadores únicos
Serie-A 2021-2022: 608 jugadores únicos
Serie-A 2022-2023: 577 jugadores únicos
Serie-A 2023-2024: 590 jugadores únicos
Serie-A 2024-2025: 599 jugadores únicos
Premier-League 2020-2021: 524 jugadores únicos
Premier-League 2021-2022: 536 jugadores únicos
Premier-League 2022-2023: 553 jugadores únicos
Premier-League 2023-2024: 569 jugadores únicos
Premier-League 2024-2025: 562 jugadores únicos
Ligue-1 2020-2021: 575 jugadores únicos
Ligue-1 2021-2022: 589 jugadores únicos
Ligue-1 2022-2023: 583 jugadores únicos
Ligue-1 2023-2024: 523 jugador